# UR5 policy training on Colab

Training only: this notebook never connects to ROS or commands a robot. Use a GPU runtime. Transfer your validated LeRobot dataset, including videos and conversion metadata. Start with SmolVLA; Pi0 may exceed the assigned GPU memory. Keep complete checkpoints on persistent storage. Commands follow the repository README.


In [ ]:
from pathlib import Path
import subprocess
import sys
REPOSITORY = Path('/content/Ur5-vla-complete-guide')
if not REPOSITORY.exists():
    subprocess.run(['git', 'clone', 'https://github.com/shafakatmasud/Ur5-vla-complete-guide.git', str(REPOSITORY)], check=True)
print(subprocess.check_output(['git', '-C', str(REPOSITORY), 'rev-parse', 'HEAD'], text=True))
subprocess.run(['nvidia-smi'], check=True)


## Isolated ML environment

Install Python 3.12 with uv; this avoids replacing the Colab kernel interpreter. The observed constraints may require adjustment if the runtime GPU/driver cannot support their PyTorch wheels. Record any change.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', str(REPOSITORY / '.venv-vla')], check=True)
VLA_PYTHON = REPOSITORY / '.venv-vla/bin/python'
subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(VLA_PYTHON),
                '-c', str(REPOSITORY / 'environments/observed-ml-constraints.txt'),
                '-e', str(REPOSITORY / 'vendor/lerobot') + '[training,smolvla,pi,diffusion]'], check=True)


## Dataset and output paths

Copy your dataset to runtime-local storage before setting these paths. Mount your own Google Drive if needed using `from google.colab import drive; drive.mount("/content/drive")`. Choose a new output directory. After each saved checkpoint, copy its entire directory to persistent storage. A runtime termination can lose unsaved progress.


In [ ]:
import os
DATASET_ROOT = Path('/content/datasets/ur5_orange_cube_v1')  # change this
DATASET_REPO_ID = 'local/ur5_orange_cube_v1'               # change this
OUTPUT_ROOT = Path('/content/outputs/ur5_smolvla_run1')     # must be new
MODEL = 'smolvla'  # smolvla, pi0, act, diffusion; see deployment limitations in README
assert (DATASET_ROOT / 'meta/info.json').is_file(), 'Copy the complete converted dataset first'
environment = os.environ.copy()
environment['PYTHONPATH'] = str(REPOSITORY / 'ros2_ws/src/ur5-vla-dataset-recorder')
environment['HF_HOME'] = '/content/huggingface_cache'


## Authentication

Use interactive login; never paste tokens into saved notebook cells. Skip HF login only if downloads are public and unauthenticated access is sufficient.


In [ ]:
from getpass import getpass
hf_token = getpass('Hugging Face token (blank to skip): ')
if hf_token:
    auth_env = environment.copy()
    auth_env['HF_TOKEN'] = hf_token
    subprocess.run([str(VLA_PYTHON), '-c', 'import os; from huggingface_hub import login; login(token=os.environ["HF_TOKEN"])'], env=auth_env, check=True)
    del auth_env
wandb_key = getpass('W&B API key: ')
if wandb_key:
    environment['WANDB_API_KEY'] = wandb_key
# Do not print environment: it may contain credentials.


## Preview and train

The dry run checks the dataset contract and prints the native command. Full training starts only when the final cell is executed. For Pi0, review GPU memory and precision requirements first.


In [ ]:
command = [str(VLA_PYTHON), '-u', str(REPOSITORY / 'scripts/train_policy.py'),
           '--model', MODEL, '--dataset', str(DATASET_ROOT), '--repo-id', DATASET_REPO_ID,
           '--output', str(OUTPUT_ROOT), '--steps', '20000', '--batch-size', '4',
           '--save-freq', '1000', '--wandb-project', 'ur5-colab']
subprocess.run(command + ['--dry-run'], env=environment, check=True)


In [ ]:
# Execute this cell to start training. W&B prints the run URL.
subprocess.run(command, env=environment, check=True)


## Preserve the run

Copy the full output directory, or complete checkpoint directories including training state, to storage you control. For inference, retain every file in pretrained_model, including processors/statistics. Native resume uses the checkpoint train_config.json with --resume=true; see README. Never connect a Colab session directly to the physical arm.
